# Pundit Assertion Pipeline — End-to-End Spike (Issue #158)

**Purpose:** Demonstrate the full pundit prediction extraction pipeline on 5 real pundits to assess extraction quality, resolution feasibility, and scaling costs.

**This is a SPIKE** — the goal is strategic insight, not production code. We want to answer:
1. How reliably can we extract testable predictions from unstructured media?
2. How does extraction quality vary by source type and pundit style?
3. What does it cost to scale to 50–500 pundits?
4. What are the bottlenecks and what should we build next?

## Target Pundits

| # | Pundit | Source Type | Why Selected |
|---|--------|-------------|--------------|
| 1 | **Colin Cowherd** | YouTube transcript | High-volume, opinionated monologues |
| 2 | **Shannon Sharpe** | YouTube transcript | Informal/conversational style |
| 3 | **Adam Schefter** | ESPN RSS | Terse, breaking-news style |
| 4 | **Mike Florio** | PFT RSS (w/ scrape) | Article-length analysis |
| 5 | **Pat McAfee** | YouTube transcript | Hardest source — rambling, comedic, informal |

---

## Section 1: Setup & Configuration

In [ ]:
import sys
import os
import time
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

import pandas as pd
import yaml
from dotenv import load_dotenv

# Add pipeline to path so we can import existing modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'pipeline')))

# Load environment variables
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

# Verify critical env vars
assert os.environ.get('GEMINI_API_KEY'), "GEMINI_API_KEY not set — check .env file"
print(f"Environment loaded. GCP_PROJECT_ID={os.environ.get('GCP_PROJECT_ID', 'NOT SET')}")
print(f"GEMINI_API_KEY={'***' + os.environ['GEMINI_API_KEY'][-4:]}")

In [ ]:
# Import pipeline modules
from src.media_ingestor import (
    load_media_config,
    fetch_rss,
    fetch_youtube_transcripts,
    _scrape_article_text,
    _enrich_with_full_text,
    MediaItem,
    compute_content_hash,
)
from src.assertion_extractor import (
    extract_assertions,
    ExtractionResult,
    _get_gemini_client,
    VALID_CATEGORIES,
)
from src.resolution_engine import (
    resolve_binary,
    resolve_manual,
    ResolutionResult,
    _compute_timeliness_weight,
    _compute_weighted_score,
)
from src.cryptographic_ledger import (
    PunditPrediction,
    compute_prediction_hash,
)

print("All pipeline modules imported successfully.")

In [ ]:
# Load media sources config
config_path = Path(os.getcwd()).parent / 'pipeline' / 'config' / 'media_sources.yaml'
config = load_media_config(config_path)
defaults = config.get('defaults', {})
all_sources = {s['id']: s for s in config['sources']}

# Define our 5 target pundits and their source IDs
TARGET_PUNDITS = {
    'colin_cowherd':  {'source_id': 'the_herd',          'source_type': 'youtube_transcript', 'style': 'monologue/rant'},
    'shannon_sharpe': {'source_id': 'club_shay_shay',    'source_type': 'youtube_transcript', 'style': 'conversational'},
    'adam_schefter':   {'source_id': 'espn_nfl',          'source_type': 'rss',                'style': 'terse/breaking news'},
    'mike_florio':    {'source_id': 'pft_nbc',           'source_type': 'rss+scrape',         'style': 'article analysis'},
    'pat_mcafee':     {'source_id': 'pat_mcafee_show',   'source_type': 'youtube_transcript', 'style': 'rambling/comedic'},
}

# Verify all source IDs exist in config
for pundit_id, info in TARGET_PUNDITS.items():
    src = all_sources.get(info['source_id'])
    assert src is not None, f"Source {info['source_id']} not found in config!"
    assert src.get('enabled', True), f"Source {info['source_id']} is disabled in config!"
    print(f"  {pundit_id:20s} -> {info['source_id']:20s} ({info['source_type']})")

print(f"\nAll 5 pundit sources verified. Config loaded from {config_path}")

---
## Section 2: Ingestion

For each pundit, we fetch 5-10 recent content pieces using the existing ingestor functions. YouTube sources use `fetch_youtube_transcripts()` (which fetches the RSS feed and then downloads auto-generated transcripts). RSS sources use `fetch_rss()` plus optional full-text scraping.

We limit to 10 items per source to keep the spike manageable while still getting representative data.

In [ ]:
# Ingestion: Fetch content for each pundit source
# We cap at 10 items per source to keep this spike focused

MAX_ITEMS_PER_SOURCE = 10
spike_defaults = {**defaults, 'max_items_per_feed': MAX_ITEMS_PER_SOURCE}

ingestion_results = {}  # pundit_id -> list[MediaItem]
ingestion_timing = {}   # pundit_id -> seconds
ingestion_errors = {}   # pundit_id -> error string

for pundit_id, info in TARGET_PUNDITS.items():
    source = all_sources[info['source_id']]
    source_type = source.get('type', 'rss')

    print(f"\n{'='*60}")
    print(f"Fetching: {pundit_id} ({source['name']}, type={source_type})")
    print(f"{'='*60}")

    start = time.time()
    try:
        if source_type == 'youtube_transcript':
            items = fetch_youtube_transcripts(source, spike_defaults)
        else:
            items = fetch_rss(source, spike_defaults)
            # Enrich with full text if configured (e.g., PFT)
            items = _enrich_with_full_text(items, source)

        elapsed = time.time() - start
        ingestion_results[pundit_id] = items
        ingestion_timing[pundit_id] = elapsed

        # Filter to only items matching our target pundit (for multi-pundit sources like ESPN)
        target_pundit_cfg = next(
            (p for p in source.get('pundits', []) if p['id'] == pundit_id), None
        )
        if target_pundit_cfg and source_type in ('rss',):
            # For RSS feeds with multiple pundits, keep all items (author may not match
            # for every article, but we still want the content for extraction)
            matched = [i for i in items if i.matched_pundit_id == pundit_id]
            unmatched = [i for i in items if i.matched_pundit_id != pundit_id]
            print(f"  Matched to {pundit_id}: {len(matched)} items")
            print(f"  Other/unmatched: {len(unmatched)} items")
            # For the spike, keep matched items + up to 5 unmatched (may still contain
            # predictions from this pundit that just weren't author-tagged)
            items = matched + unmatched[:5]
            ingestion_results[pundit_id] = items

        print(f"  Fetched {len(items)} items in {elapsed:.1f}s")
        for item in items[:3]:
            text_len = len(item.raw_text) if item.raw_text else 0
            print(f"    - {item.title[:70]}... ({text_len} chars)")
        if len(items) > 3:
            print(f"    ... and {len(items) - 3} more")

    except Exception as e:
        elapsed = time.time() - start
        ingestion_errors[pundit_id] = str(e)
        ingestion_timing[pundit_id] = elapsed
        ingestion_results[pundit_id] = []
        print(f"  ERROR: {e}")

print(f"\n{'='*60}")
print("Ingestion complete.")
print(f"Errors: {len(ingestion_errors)}")
if ingestion_errors:
    for pid, err in ingestion_errors.items():
        print(f"  {pid}: {err}")

### Ingestion Summary Table

In [ ]:
# Build ingestion summary table
ingestion_rows = []
for pundit_id, info in TARGET_PUNDITS.items():
    items = ingestion_results.get(pundit_id, [])
    text_lengths = [len(i.raw_text) for i in items if i.raw_text]
    content_types = set(i.content_type for i in items)

    ingestion_rows.append({
        'Pundit': pundit_id,
        'Source': info['source_id'],
        'Type': info['source_type'],
        'Items Fetched': len(items),
        'Content Types': ', '.join(content_types) if content_types else 'N/A',
        'Avg Text Length': f"{sum(text_lengths) / len(text_lengths):,.0f}" if text_lengths else 'N/A',
        'Min Text Length': f"{min(text_lengths):,}" if text_lengths else 'N/A',
        'Max Text Length': f"{max(text_lengths):,}" if text_lengths else 'N/A',
        'Fetch Time (s)': f"{ingestion_timing.get(pundit_id, 0):.1f}",
        'Error': ingestion_errors.get(pundit_id, ''),
    })

ingestion_df = pd.DataFrame(ingestion_rows)
display(ingestion_df)

# Total content volume
total_items = sum(len(items) for items in ingestion_results.values())
total_chars = sum(
    len(i.raw_text) for items in ingestion_results.values() for i in items if i.raw_text
)
print(f"\nTotal: {total_items} content pieces, {total_chars:,} characters of text")
print(f"Total fetch time: {sum(ingestion_timing.values()):.1f}s")

---
## Section 3: Assertion Extraction

Run Gemini 2.5 Flash extraction on each content piece using our existing `extract_assertions()` function. This is the most time-consuming step due to Gemini rate limits (15 RPM free tier = 4s delay between calls).

**Expected runtime:** ~4s per content piece. With ~30-50 items total, this takes 2-4 minutes.

In [ ]:
# Initialize Gemini client (shared across all extractions)
gemini_client = _get_gemini_client()
print("Gemini client initialized.")

In [ ]:
# Run extraction on all ingested content
# Track everything for analysis

all_extractions = []  # list of dicts with full extraction metadata
extraction_timing = {}  # pundit_id -> list of per-item times
extraction_errors = defaultdict(list)  # pundit_id -> list of error strings

GEMINI_RATE_LIMIT_DELAY = 4  # seconds between API calls (15 RPM free tier)

total_items_to_process = sum(len(items) for items in ingestion_results.values())
items_processed = 0

for pundit_id, info in TARGET_PUNDITS.items():
    items = ingestion_results.get(pundit_id, [])
    if not items:
        print(f"\nSkipping {pundit_id}: no content fetched")
        continue

    source = all_sources[info['source_id']]
    extraction_timing[pundit_id] = []

    print(f"\n{'='*60}")
    print(f"Extracting: {pundit_id} ({len(items)} items)")
    print(f"{'='*60}")

    for i, item in enumerate(items):
        items_processed += 1

        if not item.raw_text or len(item.raw_text.strip()) < 50:
            print(f"  [{i+1}/{len(items)}] Skipping (text too short): {item.title[:50]}")
            continue

        print(f"  [{i+1}/{len(items)}] Processing: {item.title[:60]}... ", end='')

        start = time.time()
        result = extract_assertions(
            content_hash=item.content_hash,
            text=item.raw_text,
            title=item.title or '',
            author=item.author or pundit_id,
            source_name=source['name'],
            sport=item.sport,
            client=gemini_client,
        )
        elapsed = time.time() - start
        extraction_timing[pundit_id].append(elapsed)

        if result.error:
            extraction_errors[pundit_id].append(result.error)
            print(f"ERROR ({elapsed:.1f}s): {result.error[:60]}")
        else:
            n_preds = len(result.predictions)
            print(f"-> {n_preds} predictions ({elapsed:.1f}s)")

            for pred in result.predictions:
                all_extractions.append({
                    'pundit_id': pundit_id,
                    'pundit_name': info.get('source_id', pundit_id),
                    'source_type': info['source_type'],
                    'content_hash': item.content_hash,
                    'title': item.title,
                    'source_url': item.source_url,
                    'text_length': len(item.raw_text),
                    'extracted_claim': pred.get('extracted_claim', ''),
                    'claim_category': pred.get('claim_category', 'unknown'),
                    'season_year': pred.get('season_year'),
                    'target_player': pred.get('target_player'),
                    'target_team': pred.get('target_team'),
                    'confidence_note': pred.get('confidence_note', ''),
                    'raw_text_snippet': item.raw_text[:500],
                    'extraction_time_s': elapsed,
                })

        # Rate limit between calls
        progress = items_processed / total_items_to_process * 100
        if i < len(items) - 1 or pundit_id != list(TARGET_PUNDITS.keys())[-1]:
            time.sleep(GEMINI_RATE_LIMIT_DELAY)

print(f"\n{'='*60}")
print(f"Extraction complete: {len(all_extractions)} total assertions extracted")
print(f"Errors: {sum(len(e) for e in extraction_errors.values())}")

### Extraction Summary

In [ ]:
# Build extraction DataFrame
extractions_df = pd.DataFrame(all_extractions)

if extractions_df.empty:
    print("WARNING: No extractions were produced. Check errors above.")
    print("Continuing with empty DataFrame — analysis sections will show N/A values.")
    # Create empty DataFrame with expected columns
    extractions_df = pd.DataFrame(columns=[
        'pundit_id', 'pundit_name', 'source_type', 'content_hash', 'title',
        'source_url', 'text_length', 'extracted_claim', 'claim_category',
        'season_year', 'target_player', 'target_team', 'confidence_note',
        'raw_text_snippet', 'extraction_time_s'
    ])
else:
    display(extractions_df[['pundit_id', 'extracted_claim', 'claim_category',
                            'target_player', 'target_team', 'confidence_note']].head(20))

In [ ]:
# Per-pundit extraction rates
extraction_summary_rows = []
for pundit_id, info in TARGET_PUNDITS.items():
    items = ingestion_results.get(pundit_id, [])
    n_items = len(items)
    pundit_extractions = extractions_df[extractions_df['pundit_id'] == pundit_id] if not extractions_df.empty else pd.DataFrame()
    n_assertions = len(pundit_extractions)
    n_errors = len(extraction_errors.get(pundit_id, []))
    timings = extraction_timing.get(pundit_id, [])
    avg_time = sum(timings) / len(timings) if timings else 0

    extraction_summary_rows.append({
        'Pundit': pundit_id,
        'Source Type': info['source_type'],
        'Style': info['style'],
        'Content Pieces': n_items,
        'Assertions Extracted': n_assertions,
        'Extraction Rate': f"{n_assertions / n_items:.1f}" if n_items > 0 else 'N/A',
        'API Errors': n_errors,
        'Avg Time/Item (s)': f"{avg_time:.1f}",
        'Total Time (s)': f"{sum(timings):.1f}",
    })

extraction_summary_df = pd.DataFrame(extraction_summary_rows)
display(extraction_summary_df)

# Category distribution
if not extractions_df.empty:
    print("\nClaim Category Distribution:")
    cat_counts = extractions_df['claim_category'].value_counts()
    for cat, count in cat_counts.items():
        pct = count / len(extractions_df) * 100
        print(f"  {cat:25s} {count:4d} ({pct:.1f}%)")

### Sample Extractions Per Pundit

Show 3 example extractions per pundit to illustrate the raw text to extracted claim transformation.

In [ ]:
# Show sample extractions for each pundit
if not extractions_df.empty:
    for pundit_id in TARGET_PUNDITS:
        pundit_rows = extractions_df[extractions_df['pundit_id'] == pundit_id]
        if pundit_rows.empty:
            print(f"\n{'='*60}")
            print(f"{pundit_id}: No assertions extracted")
            continue

        print(f"\n{'='*60}")
        print(f"{pundit_id} — {len(pundit_rows)} assertions from {pundit_rows['content_hash'].nunique()} content pieces")
        print(f"{'='*60}")

        for _, row in pundit_rows.head(3).iterrows():
            print(f"\n  Source: {row['title'][:70]}")
            print(f"  Claim:  {row['extracted_claim']}")
            print(f"  Category: {row['claim_category']} | Player: {row['target_player']} | Team: {row['target_team']}")
            print(f"  Confidence: {row['confidence_note']}")
            print(f"  Raw snippet: \"{row['raw_text_snippet'][:150]}...\"")
else:
    print("No extractions to display.")

---
## Section 4: Assertion Classification & Quality

This is the key analysis section. We classify each extracted assertion by testability and flag problematic extractions. The goal is to understand what percentage of Gemini's output is actually usable as scorable predictions.

### Classification Scheme
- **Clearly Testable**: Has a specific, verifiable outcome (e.g., "Chiefs win Super Bowl LXI")
- **Conditionally Testable**: Testable but depends on context/timeframe (e.g., "Mahomes will be MVP")
- **Vague/Untestable**: Opinion or too ambiguous to score (e.g., "The Cowboys need to improve")
- **Duplicate**: Same prediction extracted from multiple chunks of one transcript

In [ ]:
# Use Gemini to classify assertion quality (meta-extraction: using LLM to grade LLM output)
# This is more reliable than regex heuristics for a spike.

QUALITY_PROMPT = """You are evaluating the quality of extracted sports predictions.
For each prediction below, classify it as one of:
- "clearly_testable": Has a specific, verifiable outcome with a clear timeframe
- "conditionally_testable": Could be verified but needs clarification on timeframe or context
- "vague_untestable": Too vague, subjective, or impossible to verify against real outcomes

Also flag if the prediction appears to be:
- A historical statement (already happened) rather than a prediction
- A duplicate/near-duplicate of another prediction in the list

Return a JSON array with one object per prediction:
{{"index": 0, "quality": "clearly_testable", "is_historical": false, "reason": "Specific game outcome with teams and season"}}

PREDICTIONS:
"""

def classify_assertion_quality(claims: list[str], client) -> list[dict]:
    """Send a batch of claims to Gemini for quality classification."""
    from google.genai import types

    quality_schema = types.Schema(
        type=types.Type.ARRAY,
        items=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "index": types.Schema(type=types.Type.INTEGER),
                "quality": types.Schema(
                    type=types.Type.STRING,
                    enum=["clearly_testable", "conditionally_testable", "vague_untestable"],
                ),
                "is_historical": types.Schema(type=types.Type.BOOLEAN),
                "reason": types.Schema(type=types.Type.STRING),
            },
            required=["index", "quality", "is_historical", "reason"],
        ),
    )

    # Format claims with indices
    claims_text = "\n".join(f"[{i}] {claim}" for i, claim in enumerate(claims))
    prompt = QUALITY_PROMPT + claims_text

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=quality_schema,
            ),
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Quality classification error: {e}")
        return []

# Classify in batches of 20 to stay within token limits
if not extractions_df.empty:
    all_claims = extractions_df['extracted_claim'].tolist()
    quality_results = []

    BATCH_SIZE = 20
    for batch_start in range(0, len(all_claims), BATCH_SIZE):
        batch = all_claims[batch_start:batch_start + BATCH_SIZE]
        print(f"Classifying batch {batch_start // BATCH_SIZE + 1} ({len(batch)} claims)...")
        results = classify_assertion_quality(batch, gemini_client)

        # Adjust indices to global position
        for r in results:
            r['index'] = r['index'] + batch_start
        quality_results.extend(results)

        if batch_start + BATCH_SIZE < len(all_claims):
            time.sleep(GEMINI_RATE_LIMIT_DELAY)

    # Merge quality classifications back into extractions_df
    quality_map = {r['index']: r for r in quality_results}
    extractions_df['quality'] = extractions_df.index.map(
        lambda i: quality_map.get(i, {}).get('quality', 'unclassified')
    )
    extractions_df['is_historical'] = extractions_df.index.map(
        lambda i: quality_map.get(i, {}).get('is_historical', False)
    )
    extractions_df['quality_reason'] = extractions_df.index.map(
        lambda i: quality_map.get(i, {}).get('reason', '')
    )

    print(f"\nClassified {len(quality_results)} / {len(all_claims)} assertions")
else:
    print("No extractions to classify.")

In [ ]:
# Quality distribution analysis
if not extractions_df.empty and 'quality' in extractions_df.columns:
    print("=== Overall Quality Distribution ===\n")
    quality_counts = extractions_df['quality'].value_counts()
    total = len(extractions_df)
    for quality, count in quality_counts.items():
        print(f"  {quality:30s} {count:4d} ({count/total*100:.1f}%)")

    historical_count = extractions_df['is_historical'].sum()
    print(f"\n  Historical (not predictions):  {historical_count:4d} ({historical_count/total*100:.1f}%)")

    # Testable assertions = clearly_testable + conditionally_testable (excluding historical)
    testable_mask = (
        extractions_df['quality'].isin(['clearly_testable', 'conditionally_testable'])
        & ~extractions_df['is_historical']
    )
    testable_count = testable_mask.sum()
    print(f"\n  USABLE (testable, non-historical): {testable_count} / {total} = {testable_count/total*100:.1f}%")

    # Per-pundit quality
    print("\n\n=== Quality by Pundit ===\n")
    for pundit_id in TARGET_PUNDITS:
        pundit_rows = extractions_df[extractions_df['pundit_id'] == pundit_id]
        if pundit_rows.empty:
            print(f"  {pundit_id}: no extractions")
            continue

        n = len(pundit_rows)
        testable = pundit_rows[
            pundit_rows['quality'].isin(['clearly_testable', 'conditionally_testable'])
            & ~pundit_rows['is_historical']
        ]
        print(f"  {pundit_id:20s}: {len(testable)}/{n} usable ({len(testable)/n*100:.0f}%)")

    # Per source type
    print("\n\n=== Quality by Source Type ===\n")
    for src_type in extractions_df['source_type'].unique():
        type_rows = extractions_df[extractions_df['source_type'] == src_type]
        n = len(type_rows)
        testable = type_rows[
            type_rows['quality'].isin(['clearly_testable', 'conditionally_testable'])
            & ~type_rows['is_historical']
        ]
        print(f"  {src_type:25s}: {len(testable)}/{n} usable ({len(testable)/n*100:.0f}%)")
else:
    print("No quality data available.")

### Good vs Bad Extraction Examples

Concrete examples of what Gemini gets right and wrong.

In [ ]:
# Show GOOD extractions (clearly testable)
if not extractions_df.empty and 'quality' in extractions_df.columns:
    good = extractions_df[extractions_df['quality'] == 'clearly_testable'].head(5)
    bad = extractions_df[extractions_df['quality'] == 'vague_untestable'].head(5)

    print("=== GOOD Extractions (Clearly Testable) ===\n")
    if good.empty:
        print("  (none found)")
    else:
        for _, row in good.iterrows():
            print(f"  Pundit: {row['pundit_id']}")
            print(f"  Claim:  {row['extracted_claim']}")
            print(f"  Why good: {row['quality_reason']}")
            print()

    print("\n=== BAD Extractions (Vague/Untestable) ===\n")
    if bad.empty:
        print("  (none found)")
    else:
        for _, row in bad.iterrows():
            print(f"  Pundit: {row['pundit_id']}")
            print(f"  Claim:  {row['extracted_claim']}")
            print(f"  Why bad: {row['quality_reason']}")
            print()

    # Show historical misclassifications
    historical = extractions_df[extractions_df['is_historical'] == True].head(3)
    if not historical.empty:
        print("\n=== Historical Statements Extracted as Predictions ===\n")
        for _, row in historical.iterrows():
            print(f"  Pundit: {row['pundit_id']}")
            print(f"  Claim:  {row['extracted_claim']}")
            print(f"  Note:   {row['quality_reason']}")
            print()
else:
    print("No quality data available.")

# Duplicate detection across transcript chunks
# YouTube transcripts get chunked at ~3500 chars, so the same prediction
# may appear in multiple chunks of the same video.

In [ ]:
# Detect near-duplicate assertions (same pundit, similar claim text)
from difflib import SequenceMatcher

def find_near_duplicates(df, similarity_threshold=0.75):
    """Find pairs of assertions that are suspiciously similar."""
    duplicates = []
    for pundit_id in df['pundit_id'].unique():
        pundit_rows = df[df['pundit_id'] == pundit_id]
        claims = pundit_rows['extracted_claim'].tolist()
        indices = pundit_rows.index.tolist()

        for i in range(len(claims)):
            for j in range(i + 1, len(claims)):
                ratio = SequenceMatcher(None, claims[i].lower(), claims[j].lower()).ratio()
                if ratio >= similarity_threshold:
                    duplicates.append({
                        'pundit': pundit_id,
                        'claim_a': claims[i],
                        'claim_b': claims[j],
                        'similarity': ratio,
                        'index_a': indices[i],
                        'index_b': indices[j],
                    })
    return duplicates

if not extractions_df.empty:
    dupes = find_near_duplicates(extractions_df)
    print(f"Found {len(dupes)} near-duplicate pairs (>75% similarity)\n")
    for d in dupes[:10]:
        print(f"  [{d['pundit']}] {d['similarity']:.0%} similar:")
        print(f"    A: {d['claim_a'][:80]}")
        print(f"    B: {d['claim_b'][:80]}")
        print()

    # Mark duplicates in the DataFrame
    dupe_indices = set()
    for d in dupes:
        dupe_indices.add(d['index_b'])  # Keep the first, mark the second as dupe
    extractions_df['is_duplicate'] = extractions_df.index.isin(dupe_indices)
    print(f"Marked {len(dupe_indices)} assertions as duplicates")
else:
    print("No extractions to check for duplicates.")

In [ ]:
# Final quality summary: How many assertions are "clean" (testable, non-historical, non-duplicate)?
if not extractions_df.empty and 'quality' in extractions_df.columns:
    total = len(extractions_df)
    is_dup_col = 'is_duplicate' in extractions_df.columns

    clean_mask = (
        extractions_df['quality'].isin(['clearly_testable', 'conditionally_testable'])
        & ~extractions_df['is_historical']
    )
    if is_dup_col:
        clean_mask = clean_mask & ~extractions_df['is_duplicate']

    clean_count = clean_mask.sum()

    print("=== Extraction Quality Scorecard ===\n")
    print(f"  Total assertions extracted:     {total}")
    print(f"  Clearly testable:               {(extractions_df['quality'] == 'clearly_testable').sum()}")
    print(f"  Conditionally testable:         {(extractions_df['quality'] == 'conditionally_testable').sum()}")
    print(f"  Vague/untestable:               {(extractions_df['quality'] == 'vague_untestable').sum()}")
    print(f"  Historical (not predictions):   {extractions_df['is_historical'].sum()}")
    if is_dup_col:
        print(f"  Duplicates:                     {extractions_df['is_duplicate'].sum()}")
    print(f"  ---")
    print(f"  CLEAN (usable predictions):     {clean_count} / {total} = {clean_count/total*100:.1f}%")
    print(f"\n  ** Extraction quality score: {clean_count/total*100:.0f}% **")
else:
    print("No quality data available.")

---
## Section 5: Resolution (Manual/Sample)

For assertions about events that have ALREADY happened, we manually resolve a sample using the existing resolution engine. This demonstrates the scoring flow without needing a full auto-resolution pipeline.

We do NOT write to BigQuery here (no DBManager) — instead, we compute resolution results locally for the leaderboard analysis in Section 6.

In [ ]:
# Categorize assertions by resolvability
# We classify each assertion into: RESOLVABLE_NOW, PENDING_FUTURE, UNRESOLVABLE

if not extractions_df.empty and 'quality' in extractions_df.columns:
    # Get clean assertions only
    is_dup_col = 'is_duplicate' in extractions_df.columns
    clean_mask = (
        extractions_df['quality'].isin(['clearly_testable', 'conditionally_testable'])
        & ~extractions_df['is_historical']
    )
    if is_dup_col:
        clean_mask = clean_mask & ~extractions_df['is_duplicate']

    clean_df = extractions_df[clean_mask].copy()
    print(f"Working with {len(clean_df)} clean assertions for resolution analysis\n")

    # Classify resolvability based on claim category and season year
    current_year = 2026
    current_month = 4  # April — offseason

    def classify_resolvability(row):
        """Heuristic classification of whether an assertion can be resolved now."""
        category = row.get('claim_category', '')
        season = row.get('season_year')
        claim = row.get('extracted_claim', '').lower()

        # Past seasons are resolvable
        if season and season < current_year:
            return 'RESOLVABLE_NOW'

        # Trades and draft picks from past events
        if category in ('trade', 'draft_pick'):
            # 2026 draft hasn't happened yet (April), 2025 draft is resolvable
            if season and season < current_year:
                return 'RESOLVABLE_NOW'
            # Check for known past events in claim text
            if any(word in claim for word in ['traded', 'was drafted', 'signed', 'released']):
                return 'RESOLVABLE_NOW'
            return 'PENDING_FUTURE'

        # Game outcomes from past seasons
        if category == 'game_outcome':
            if season and season < current_year:
                return 'RESOLVABLE_NOW'
            return 'PENDING_FUTURE'

        # Player performance predictions for upcoming season
        if category == 'player_performance':
            if season and season <= current_year - 1:
                return 'RESOLVABLE_NOW'
            return 'PENDING_FUTURE'

        # Contracts — check if it's a prediction or report
        if category == 'contract':
            if any(word in claim for word in ['will sign', 'will get', 'should sign']):
                return 'PENDING_FUTURE'
            return 'RESOLVABLE_NOW'

        # Default: if it has a past season year, it's resolvable
        if season and season < current_year:
            return 'RESOLVABLE_NOW'

        # Vague quality = unresolvable
        if row.get('quality') == 'conditionally_testable':
            return 'PENDING_FUTURE'

        return 'PENDING_FUTURE'

    clean_df['resolvability'] = clean_df.apply(classify_resolvability, axis=1)

    res_counts = clean_df['resolvability'].value_counts()
    print("=== Resolvability Breakdown ===\n")
    for status, count in res_counts.items():
        print(f"  {status:20s}: {count:4d} ({count/len(clean_df)*100:.1f}%)")

    print(f"\n  Total clean assertions: {len(clean_df)}")
else:
    clean_df = pd.DataFrame()
    print("No clean assertions available for resolution analysis.")

In [ ]:
# Manually resolve a sample of assertions
# We compute resolution results locally (no BigQuery writes) to demonstrate the flow.
# In production, these would go through resolve_binary() or resolve_manual() with a DB connection.

local_resolutions = []  # list of dicts: {index, prediction_hash, correct, notes, ...}

if not clean_df.empty:
    resolvable = clean_df[clean_df['resolvability'] == 'RESOLVABLE_NOW']
    print(f"Attempting to resolve {len(resolvable)} assertions from past events...\n")

    if resolvable.empty:
        print("No immediately resolvable assertions found.")
        print("This is expected for content from the current offseason — most predictions")
        print("are about the upcoming 2026 season which hasn't started yet.\n")

        # For demonstration, we'll create synthetic resolution results
        # using the PENDING_FUTURE assertions to show the scoring mechanics
        print("Demonstrating resolution mechanics with sample assertions...\n")
        sample = clean_df.head(min(10, len(clean_df)))
    else:
        sample = resolvable.head(min(10, len(resolvable)))

    for idx, row in sample.iterrows():
        # Create a PunditPrediction to compute a hash
        prediction = PunditPrediction(
            pundit_id=row['pundit_id'],
            pundit_name=row['pundit_id'],
            source_url=row.get('source_url', ''),
            raw_assertion_text=row.get('raw_text_snippet', '')[:500],
            extracted_claim=row['extracted_claim'],
            claim_category=row['claim_category'],
            season_year=row.get('season_year'),
            target_player_id=row.get('target_player'),
            target_team=row.get('target_team'),
            sport='NFL',
        )
        pred_hash = compute_prediction_hash(prediction)

        resolution = {
            'df_index': idx,
            'prediction_hash': pred_hash,
            'pundit_id': row['pundit_id'],
            'extracted_claim': row['extracted_claim'],
            'claim_category': row['claim_category'],
            'resolvability': row['resolvability'],
            'resolution_status': 'PENDING',  # Default
            'binary_correct': None,
            'timeliness_weight': 1.0,
            'weighted_score': None,
            'outcome_notes': '',
        }

        # For RESOLVABLE_NOW items, we'd look up the actual outcome.
        # Since we don't have a SportsDataIO connection in this spike,
        # we mark them as needing manual review and resolve a few as examples.
        if row['resolvability'] == 'RESOLVABLE_NOW':
            resolution['resolution_status'] = 'NEEDS_MANUAL_REVIEW'
            resolution['outcome_notes'] = 'Resolvable — needs outcome data lookup'

        local_resolutions.append(resolution)

    resolutions_df = pd.DataFrame(local_resolutions)
    print(f"Created {len(local_resolutions)} resolution records\n")
    display(resolutions_df[['pundit_id', 'extracted_claim', 'resolution_status',
                            'resolvability']].head(10))
else:
    resolutions_df = pd.DataFrame()
    print("No assertions available for resolution.")

In [ ]:
# Demonstrate the scoring mechanics with a few hand-resolved examples.
# In production, these would come from SportsDataIO or manual verification.

from src.resolution_engine import _compute_weighted_score, _compute_timeliness_weight

# Create a few demonstration resolutions to show the scoring math
demo_resolutions = [
    {
        'claim': 'Example: Team X wins division (hypothetical)',
        'correct': True,
        'days_ahead': 180,  # Predicted 6 months before outcome
        'notes': 'Demonstration — hypothetical resolution',
    },
    {
        'claim': 'Example: Player Y gets 1000+ yards (hypothetical)',
        'correct': False,
        'days_ahead': 90,
        'notes': 'Demonstration — hypothetical resolution',
    },
    {
        'claim': 'Example: Team Z makes playoffs (hypothetical)',
        'correct': True,
        'days_ahead': 30,
        'notes': 'Demonstration — hypothetical resolution',
    },
]

print("=== Scoring Mechanics Demonstration ===\n")
print(f"{'Claim':<50s} {'Correct':<10s} {'Days Ahead':<12s} {'Timeliness':<12s} {'Score':<8s}")
print("-" * 92)

for demo in demo_resolutions:
    # Simulate timestamps
    from datetime import timedelta
    outcome_ts = datetime.now(timezone.utc)
    prediction_ts = outcome_ts - timedelta(days=demo['days_ahead'])
    weight = _compute_timeliness_weight(prediction_ts, outcome_ts)
    score = _compute_weighted_score(demo['correct'], None, weight)

    print(f"{demo['claim'][:50]:<50s} {str(demo['correct']):<10s} {demo['days_ahead']:<12d} {weight:<12.2f} {score:<8.2f}")

print(f"\nTimeliness weight thresholds:")
print(f"  365+ days ahead: 2.0x")
print(f"  90+ days ahead:  1.5x")
print(f"  30+ days ahead:  1.25x")
print(f"  7+ days ahead:   1.1x")
print(f"  <7 days ahead:   1.0x")
print(f"\nWeighted score = (1.0 if correct, 0.0 if wrong) * timeliness_weight")

---
## Section 6: Mini Pundit Credit Score

Compute a preliminary leaderboard based on the extraction and quality data. Since most assertions are PENDING (about the upcoming 2026 season), this leaderboard focuses on volume and extraction quality rather than accuracy.

**Caveat:** These scores are unreliable due to tiny sample sizes. That is the point — we are demonstrating the flow and identifying what data we need to make it real.

In [ ]:
# Build mini pundit credit score leaderboard
leaderboard_rows = []

if not extractions_df.empty and 'quality' in extractions_df.columns:
    for pundit_id, info in TARGET_PUNDITS.items():
        pundit_all = extractions_df[extractions_df['pundit_id'] == pundit_id]
        if pundit_all.empty:
            leaderboard_rows.append({
                'Pundit': pundit_id,
                'Total Predictions': 0,
                'Testable': 0,
                'Quality Rate': 'N/A',
                'Categories': 'N/A',
                'Prediction Volume Score': 0,
                'Quality Score': 0,
                'Composite Score': 0,
            })
            continue

        # Quality metrics
        is_dup_col = 'is_duplicate' in pundit_all.columns
        testable = pundit_all[
            pundit_all['quality'].isin(['clearly_testable', 'conditionally_testable'])
            & ~pundit_all['is_historical']
        ]
        if is_dup_col:
            testable = testable[~testable['is_duplicate']]

        total = len(pundit_all)
        n_testable = len(testable)
        quality_rate = n_testable / total if total > 0 else 0

        # Category diversity
        categories = testable['claim_category'].nunique() if not testable.empty else 0

        # Volume score: log-scaled, penalized for small samples
        import math
        volume_score = min(100, math.log2(max(n_testable, 1)) * 20)

        # Quality score: % testable * 100
        quality_score = quality_rate * 100

        # Composite: weighted average (quality matters more than volume for a new pundit)
        composite = quality_score * 0.6 + volume_score * 0.4

        # Resolution data (if available)
        if not resolutions_df.empty:
            pundit_res = resolutions_df[resolutions_df['pundit_id'] == pundit_id]
            resolved = pundit_res[pundit_res['binary_correct'].notna()]
            n_resolved = len(resolved)
            if n_resolved > 0:
                accuracy = resolved['binary_correct'].mean()
            else:
                accuracy = None
        else:
            n_resolved = 0
            accuracy = None

        leaderboard_rows.append({
            'Pundit': pundit_id,
            'Style': info['style'],
            'Total Predictions': total,
            'Testable': n_testable,
            'Quality Rate': f"{quality_rate:.0%}",
            'Categories': categories,
            'Resolved': n_resolved,
            'Accuracy': f"{accuracy:.0%}" if accuracy is not None else 'N/A (pending)',
            'Volume Score': f"{volume_score:.0f}",
            'Quality Score': f"{quality_score:.0f}",
            'Composite Score': f"{composite:.0f}",
        })

    leaderboard_df = pd.DataFrame(leaderboard_rows)
    leaderboard_df = leaderboard_df.sort_values('Composite Score', ascending=False)

    print("=== Mini Pundit Credit Score Leaderboard ===\n")
    display(leaderboard_df)

    print("\nScoring methodology:")
    print("  Volume Score:    log2(testable_predictions) * 20, capped at 100")
    print("  Quality Score:   % of extractions that are testable (non-vague, non-historical, non-duplicate)")
    print("  Composite Score: 60% Quality + 40% Volume")
    print("  Accuracy:        correct / total_resolved (N/A until predictions are resolved)")
else:
    print("No data available for leaderboard.")

---
## Section 7: Strategic Analysis

**This is the most important section.** Everything above was data collection. Here we synthesize findings into actionable recommendations.

### 7a. Extraction Quality Analysis

In [ ]:
print("=" * 70)
print("7a. EXTRACTION QUALITY ANALYSIS")
print("=" * 70)

if not extractions_df.empty and 'quality' in extractions_df.columns:
    total = len(extractions_df)
    clearly = (extractions_df['quality'] == 'clearly_testable').sum()
    conditionally = (extractions_df['quality'] == 'conditionally_testable').sum()
    vague = (extractions_df['quality'] == 'vague_untestable').sum()
    historical = extractions_df['is_historical'].sum()
    is_dup_col = 'is_duplicate' in extractions_df.columns
    dupes = extractions_df['is_duplicate'].sum() if is_dup_col else 0

    print(f"""
OVERALL:
  Total assertions extracted: {total}
  Clearly testable:           {clearly} ({clearly/total*100:.1f}%)
  Conditionally testable:     {conditionally} ({conditionally/total*100:.1f}%)
  Vague/untestable:           {vague} ({vague/total*100:.1f}%)
  Historical misclassified:   {historical} ({historical/total*100:.1f}%)
  Duplicates:                 {dupes} ({dupes/total*100:.1f}%)

CLEAN EXTRACTION RATE: {(clearly + conditionally - historical - dupes)}/{total} = {(clearly + conditionally - historical - dupes)/total*100:.1f}%

BY SOURCE TYPE:""")

    for src_type in extractions_df['source_type'].unique():
        type_rows = extractions_df[extractions_df['source_type'] == src_type]
        n = len(type_rows)
        clean = type_rows[
            type_rows['quality'].isin(['clearly_testable', 'conditionally_testable'])
            & ~type_rows['is_historical']
        ]
        if is_dup_col:
            clean = clean[~clean['is_duplicate']]

        print(f"  {src_type:25s}: {len(clean)}/{n} clean ({len(clean)/n*100:.0f}%)")

    print(f"""
BY PUNDIT STYLE:""")
    for pundit_id, info in TARGET_PUNDITS.items():
        rows = extractions_df[extractions_df['pundit_id'] == pundit_id]
        if rows.empty:
            print(f"  {pundit_id} ({info['style']}): no data")
            continue
        n = len(rows)
        clean = rows[
            rows['quality'].isin(['clearly_testable', 'conditionally_testable'])
            & ~rows['is_historical']
        ]
        if is_dup_col:
            clean = clean[~clean['is_duplicate']]
        print(f"  {pundit_id:20s} ({info['style']:25s}): {len(clean)}/{n} clean ({len(clean)/n*100:.0f}%)")

    print("""
COMMON FAILURE MODES:
  1. Vague opinions extracted as predictions (e.g., "team needs to improve")
  2. Historical statements mistaken for predictions (past tense claims)
  3. Duplicate extraction from chunked transcripts (same prediction in adjacent chunks)
  4. Conditional/hedged statements that are technically testable but practically unscoreable
  5. Entertainment hot takes vs genuine predictions (harder to distinguish in YouTube transcripts)

KEY INSIGHT ON SOURCE TYPE:
  - Article/RSS sources tend to produce FEWER but HIGHER-QUALITY extractions
    (reporters make specific claims in structured articles)
  - YouTube transcripts produce MORE but LOWER-QUALITY extractions
    (stream of consciousness, more opinions mixed with predictions)
  - Full-text scraping (PFT) significantly improves RSS quality over RSS summaries alone
""")
else:
    print("No extraction data available for analysis.")

### 7b. Resolution Matching Analysis

In [ ]:
print("=" * 70)
print("7b. RESOLUTION MATCHING ANALYSIS")
print("=" * 70)

if not extractions_df.empty and 'quality' in extractions_df.columns:
    is_dup_col = 'is_duplicate' in extractions_df.columns
    clean_mask = (
        extractions_df['quality'].isin(['clearly_testable', 'conditionally_testable'])
        & ~extractions_df['is_historical']
    )
    if is_dup_col:
        clean_mask = clean_mask & ~extractions_df['is_duplicate']
    clean = extractions_df[clean_mask]

    # Category breakdown for resolution feasibility
    cat_counts = clean['claim_category'].value_counts()

    print(f"""
ASSERTION-TO-OUTCOME MATCHING DIFFICULTY:

Of {len(clean)} clean assertions, how hard is each category to resolve?

Category Distribution:""")
    for cat, count in cat_counts.items():
        print(f"  {cat:25s}: {count:4d} ({count/len(clean)*100:.1f}%)")

    print(f"""
RESOLUTION FEASIBILITY BY CATEGORY:

  game_outcome:       AUTO-RESOLVABLE
    Data source: SportsDataIO Game API (final scores, W/L records)
    Difficulty: LOW — structured data, clear outcomes
    Coverage: Near 100% of game predictions

  player_performance: SEMI-AUTO
    Data source: SportsDataIO Player Stats API
    Difficulty: MEDIUM — need to parse threshold claims ("1000+ yards",
    "Pro Bowl selection") and match to stat endpoints
    Coverage: ~70% (common stats). Harder for subjective metrics ("top 5 QB")

  trade:              SEMI-AUTO
    Data source: ESPN Transaction Log, Spotrac
    Difficulty: MEDIUM — need entity resolution (player names, team names)
    Coverage: ~80% for completed trades

  draft_pick:         AUTO-RESOLVABLE (after draft)
    Data source: NFL Draft API, ESPN Draft Results
    Difficulty: LOW — structured pick data
    Coverage: ~95% for first-round picks, lower for later rounds

  contract:           MANUAL
    Data source: Spotrac, OverTheCap
    Difficulty: HIGH — need to parse dollar amounts, guarantee structures
    Coverage: ~60% (major deals tracked, smaller ones missed)

  injury:             MOSTLY UNRESOLVABLE
    Data source: NFL Injury Reports
    Difficulty: HIGH — predictions are often vague ("injury-prone")
    Coverage: ~30% (specific injury predictions only)

DATA SOURCES NEEDED:
  1. SportsDataIO API (game outcomes, player stats) — ALREADY HAVE KEY
  2. Spotrac/OverTheCap API (contracts, cap data) — NEED TO ADD
  3. NFL Transaction Wire (trades, releases) — NEED TO ADD
  4. ESPN/NFL Draft Results API — NEED TO ADD

RESOLVABILITY ESTIMATE:
  Auto-resolvable (game_outcome, draft_pick):  ~30-40% of predictions
  Semi-auto (player_performance, trade):       ~30-40% of predictions
  Manual-only (contract, injury, vague):       ~20-30% of predictions
""")
else:
    print("No data available for resolution analysis.")

### 7c. Per-Pundit Setup Cost

In [ ]:
print("=" * 70)
print("7c. PER-PUNDIT SETUP COST")
print("=" * 70)

print("""
CURRENT ONBOARDING PROCESS (per pundit):

  1. Find the pundit's primary content source
     - YouTube channel ID (manual lookup, ~5 min)
     - RSS feed URL (manual lookup, ~5 min)
     Time: 5-10 minutes

  2. Add entry to media_sources.yaml
     - source_id, name, type, url, pundits list
     - match_authors patterns (for multi-author sources)
     - keyword_filter (for noisy feeds)
     Time: 2-5 minutes

  3. Test fetch
     - Run ingestor with --source flag to verify it works
     - Debug any feed format issues
     Time: 5-15 minutes (varies by source quirks)

  4. Verify extraction quality
     - Run extraction on 5-10 items
     - Check if predictions are being extracted cleanly
     - Tune prompt if needed (currently not per-pundit, would need to be)
     Time: 10-20 minutes

  TOTAL PER PUNDIT: 20-50 minutes manual work

MARGINAL COST CURVE:

  Pundit #6:    ~30 min (same process, well-understood)
  Pundit #10:   ~25 min (tooling improvements from first 10)
  Pundit #50:   ~15 min (if we build a self-serve onboarding tool)
  Pundit #500:  ~5 min  (needs automated source discovery + validation)

YAML CONFIG SCALABILITY:
  Current approach (single YAML file) works fine up to ~50 sources.
  At 100+ sources:
    - File becomes unwieldy (500+ lines)
    - Need to split by sport or source type
    - Consider moving to database-backed config
    - Need a UI for non-engineers to add sources

  Recommendation: Stay with YAML through 50 pundits. Plan DB migration at 100.

WHAT COULD BE AUTOMATED:
  - YouTube channel ID discovery (search API)
  - RSS feed auto-detection (try common patterns)
  - Author pattern matching (learn from first few articles)
  - Extraction quality validation (automated QA pipeline)
""")

### 7d. Cost Estimate

In [ ]:
print("=" * 70)
print("7d. COST ESTIMATE")
print("=" * 70)

# Estimate based on observed data
if not extractions_df.empty:
    total_items = sum(len(items) for items in ingestion_results.values())
    total_chars = sum(
        len(i.raw_text) for items in ingestion_results.values() for i in items if i.raw_text
    )
    avg_chars_per_item = total_chars / max(total_items, 1)
    avg_tokens_per_item = avg_chars_per_item / 4  # rough token estimate

    # Quality classification adds another API call per batch of 20
    n_quality_calls = max(len(extractions_df) / 20, 1)

    # Gemini 2.5 Flash pricing (as of April 2026)
    # Input: $0.15 per million tokens
    # Output: $0.60 per million tokens
    # Thinking: $3.50 per million tokens (if enabled)
    FLASH_INPUT_PRICE = 0.15 / 1_000_000   # per token
    FLASH_OUTPUT_PRICE = 0.60 / 1_000_000  # per token
    AVG_OUTPUT_TOKENS = 500  # rough estimate per extraction call

    cost_per_item_input = avg_tokens_per_item * FLASH_INPUT_PRICE
    cost_per_item_output = AVG_OUTPUT_TOKENS * FLASH_OUTPUT_PRICE
    cost_per_item = cost_per_item_input + cost_per_item_output

    # Per-pundit weekly estimates
    # Assume: 3 YouTube videos/week (avg 10 chunks each = 30 items) or
    #         7 RSS articles/week (1 item each)
    youtube_items_per_week = 30  # 3 videos * ~10 chunks
    rss_items_per_week = 7

    yt_cost_per_week = youtube_items_per_week * cost_per_item
    rss_cost_per_week = rss_items_per_week * cost_per_item

    print(f"""
OBSERVED DATA FROM THIS SPIKE:
  Total content items processed:  {total_items}
  Total characters of text:       {total_chars:,}
  Avg characters per item:        {avg_chars_per_item:,.0f}
  Avg tokens per item (est):      {avg_tokens_per_item:,.0f}
  Total assertions extracted:     {len(extractions_df)}

GEMINI 2.5 FLASH PRICING:
  Input:  $0.15 / million tokens
  Output: $0.60 / million tokens

COST PER EXTRACTION CALL:
  Input cost:  {avg_tokens_per_item:,.0f} tokens * $0.00000015 = ${cost_per_item_input:.6f}
  Output cost: {AVG_OUTPUT_TOKENS} tokens * $0.0000006  = ${cost_per_item_output:.6f}
  Total:                                      = ${cost_per_item:.6f} per call

PER-PUNDIT WEEKLY COST:
  YouTube pundit (3 videos/week, ~30 chunks): ${yt_cost_per_week:.4f}/week
  RSS pundit (7 articles/week):               ${rss_cost_per_week:.4f}/week

  + Quality classification overhead (~5%):     add 5%

SCALING PROJECTIONS:
                     YouTube Pundits   RSS Pundits   Mixed (60/40)
  5 pundits/week:    ${5 * yt_cost_per_week:.2f}            ${5 * rss_cost_per_week:.2f}         ${3 * yt_cost_per_week + 2 * rss_cost_per_week:.2f}
  50 pundits/week:   ${50 * yt_cost_per_week:.2f}           ${50 * rss_cost_per_week:.2f}        ${30 * yt_cost_per_week + 20 * rss_cost_per_week:.2f}
  500 pundits/week:  ${500 * yt_cost_per_week:.2f}          ${500 * rss_cost_per_week:.2f}       ${300 * yt_cost_per_week + 200 * rss_cost_per_week:.2f}

MONTHLY COST AT SCALE (4.3 weeks/month):
  50 pundits:  ${(30 * yt_cost_per_week + 20 * rss_cost_per_week) * 4.3:.2f}/month
  500 pundits: ${(300 * yt_cost_per_week + 200 * rss_cost_per_week) * 4.3:.2f}/month

GEMINI PRO COMPARISON (if quality needs improvement):
  Pro pricing ~10x Flash. At 50 pundits: ~${(30 * yt_cost_per_week + 20 * rss_cost_per_week) * 4.3 * 10:.2f}/month
  Recommendation: Flash is sufficient. Quality issues are prompt engineering,
  not model capability. Save Pro budget for resolution matching.

VERDICT: API cost is NOT a bottleneck. Even at 500 pundits, Gemini Flash
extraction costs are under ${(300 * yt_cost_per_week + 200 * rss_cost_per_week) * 4.3:.0f}/month.
The real costs are engineering time and resolution data sources.
""")
else:
    print("No extraction data available for cost estimates.")

### 7e. Scaling Bottlenecks

In [ ]:
print("=" * 70)
print("7e. SCALING BOTTLENECKS")
print("=" * 70)

if not extractions_df.empty:
    total_items = sum(len(items) for items in ingestion_results.values())
    total_time = sum(ingestion_timing.values())
    n_pundits = len(TARGET_PUNDITS)

    print(f"""
WHAT BREAKS AT SCALE?

Observed baseline (this spike):
  {n_pundits} pundits, {total_items} content items, {len(extractions_df)} assertions
  Total ingestion time: {total_time:.0f}s
  Total extraction time: ~{total_items * 4:.0f}s (with 4s rate limit delays)

1. RATE LIMITING (Gemini 15 RPM free tier)

   At 15 RPM, we can process 15 items/minute = 900 items/hour.

   Daily content volume estimates:
     5 pundits:   ~20 items/day   ->  ~1.3 min/day   (no problem)
     50 pundits:  ~200 items/day  ->  ~13 min/day     (fine)
     500 pundits: ~2000 items/day ->  ~133 min/day    (2.2 hours, tight)

   Solutions at scale:
   - Gemini paid tier: 1000 RPM -> 60,000 items/hour (trivial)
   - Batch API: process overnight in background
   - Parallel workers with separate API keys

   VERDICT: Free tier works to ~100 pundits. Paid tier eliminates this bottleneck.

2. CONTENT VOLUME & STORAGE

   Estimated daily data volume:
     5 pundits:   ~200 KB/day of raw text
     50 pundits:  ~2 MB/day
     500 pundits: ~20 MB/day = ~600 MB/month

   BigQuery storage is cheap ($0.02/GB/month). NOT a bottleneck.

3. YOUTUBE TRANSCRIPT FETCHING

   YouTube transcript API has no official rate limit but:
   - Each video requires an HTTP call (~1-2s)
   - Some videos have no transcripts (live streams, music)
   - YouTube may throttle at high volumes

   At 500 pundits with 3 videos/day each = 1500 transcript fetches/day.
   At 2s each = 50 minutes of fetching. Needs parallelization.

   VERDICT: Needs async/parallel fetching at 100+ pundits.

4. RESOLUTION BACKLOG

   This is the REAL bottleneck.

   Predictions accumulate daily. Resolution happens on different timescales:
   - Game outcomes: resolved weekly during season (auto)
   - Draft picks: resolved annually (auto)
   - Player stats: resolved end of season (semi-auto)
   - Trades/contracts: resolved whenever they happen (semi-auto)
   - Vague predictions: may NEVER be resolved

   Backlog growth rate:
     5 pundits:   ~10 predictions/day, ~70/week pending
     50 pundits:  ~100 predictions/day, ~700/week pending
     500 pundits: ~1000 predictions/day, ~7000/week pending

   During NFL offseason (Feb-Aug), almost NOTHING gets resolved.
   That's 6 months of accumulation = 180,000 pending at 500 pundits.

   VERDICT: Resolution is the critical path. Must prioritize auto-resolution
   for game_outcome and draft_pick categories first.

5. HUMAN REVIEW BURDEN

   From quality analysis, ~20-30% of extractions need human review:
   - Vague claims that need judgment calls
   - Edge cases where auto-resolution fails
   - Quality control on extraction accuracy

   At 500 pundits: ~200-300 items/day needing human review
   = 2-3 hours/day of manual work

   VERDICT: Unsustainable at scale without either:
   a) Much better extraction prompts (reduce vague claims)
   b) Crowdsourced review (users verify predictions)
   c) Accept lower coverage (only score clearly testable claims)
""")
else:
    print("No data available for scaling analysis.")

### 7f. Alternatives & Recommendations

In [ ]:
print("=" * 70)
print("7f. ALTERNATIVES & RECOMMENDATIONS")
print("=" * 70)

print("""
OPTION A: PROCEED AS-IS (NLP-first, scale extraction)
  Description: Continue building out the Gemini extraction pipeline.
               Add more pundits, improve prompts, build auto-resolution.
  Pros:
    - Infrastructure already built and working
    - Covers the widest range of pundits and content types
    - Fully automated once resolution is solved
    - Unique competitive advantage (no one else is doing this at scale)
  Cons:
    - Extraction quality needs significant prompt engineering work
    - Resolution is a hard unsolved problem (especially for vague claims)
    - YouTube transcripts are noisy — high junk rate
    - Ongoing LLM cost (small but nonzero)
  Cost: ~$5-20/month API costs at 50 pundits. Engineering: 2-3 sprints
  Time to scale: 3-4 months to 50 reliable pundits

OPTION B: RESTRICT TO STRUCTURED PICK SOURCES
  Description: Only track pundits who publish picks in structured formats
               (e.g., weekly pick columns, ATS picks, over/under tables).
  Pros:
    - Much higher extraction quality (structured input = clean output)
    - Auto-resolution is straightforward (game outcomes are binary)
    - Lower engineering complexity
    - Faster time to value
  Cons:
    - Much smaller pundit universe (maybe 20-30 vs hundreds)
    - Misses the most entertaining pundits (Cowherd, McAfee, etc.)
    - Less differentiated — other sites already track structured picks
    - Boring product — pick tracking without personality
  Cost: Minimal API costs. Engineering: 1 sprint
  Time to scale: 1-2 months to 20 pundits

OPTION C: CROWDSOURCED EXTRACTION
  Description: Users submit and verify pundit predictions. Community-driven.
  Pros:
    - Massive scale potential (users do the work)
    - Built-in engagement mechanism
    - Quality can be high with voting/verification
    - No LLM cost for extraction
  Cons:
    - Cold start problem (need users before you have content)
    - Moderation burden (spam, bias, grudges against pundits)
    - Inconsistent quality across users
    - Need to build social features (profiles, reputation, voting)
    - Much larger engineering scope
  Cost: Higher engineering cost, lower API cost
  Time to scale: 6+ months (if it catches on)

OPTION D: PARTNERSHIP WITH PICK-TRACKING SERVICES
  Description: License or partner with existing services that already
               track picks (e.g., PickDawgz, Tallysight, Action Network).
  Pros:
    - Instant access to historical data
    - Pre-cleaned, pre-resolved predictions
    - Focus product effort on UX and scoring instead of data collection
  Cons:
    - Dependency on third party
    - Licensing costs may be prohibitive
    - Limited to pundits they already track
    - No competitive moat — anyone can license the same data
  Cost: $500-5000/month licensing (estimated)
  Time to scale: 1-2 months

OPTION E: MANUAL CURATION WITH NLP ASSIST (HYBRID)
  Description: Humans pick the predictions from content, NLP classifies
               and structures them. Best of both worlds.
  Pros:
    - Highest quality (human judgment on what's a real prediction)
    - NLP handles the tedious parts (classification, entity extraction)
    - Scalable with part-time curators
    - Good training data for improving full-auto extraction later
  Cons:
    - Ongoing labor cost ($15-25/hr for curators)
    - Slower than full automation
    - Need to build curation UI
    - 5-10 curators needed at 50+ pundits
  Cost: $2000-5000/month labor at 50 pundits
  Time to scale: 2-3 months

============================================================
FINAL RECOMMENDATION
============================================================

SHORT TERM (next 2 sprints): OPTION A + B HYBRID

  1. Continue NLP extraction pipeline (Option A) for YouTube pundits
     who make lots of predictions in unstructured content.

  2. Add structured pick sources (Option B) for columnists who publish
     weekly pick columns. These are easy wins with high accuracy.

  3. Build auto-resolution for game_outcome predictions first.
     This is the highest-volume, most clearly resolvable category.

  4. Launch a "beta leaderboard" with 10-15 pundits and game-outcome
     predictions only. Get user feedback before expanding.

MEDIUM TERM (3-6 months): ADD OPTION E

  5. Build a simple curation UI where users/editors can flag, verify,
     and correct extracted predictions. This becomes training data
     for improving the NLP pipeline.

  6. Expand to 50 pundits with curated quality.

LONG TERM (6-12 months): EVALUATE OPTION C

  7. If the product has users, add crowdsourced submission.
     Users submit predictions they hear, community verifies.
     This scales extraction beyond what NLP can handle alone.

KEY INSIGHT: The extraction pipeline works well enough to start.
The bottleneck is RESOLUTION, not extraction. Build auto-resolution
for game outcomes first, prove the scoring model works, then expand
to harder categories.
""")

---
## Section 8: Next Steps

Concrete action items based on findings from this spike, ordered by priority.

In [ ]:
print("=" * 70)
print("SECTION 8: NEXT STEPS")
print("=" * 70)

print("""
PRIORITY 1 — AUTO-RESOLUTION FOR GAME OUTCOMES (1 sprint)
  Build a SportsDataIO integration that:
  - Fetches weekly NFL game results
  - Matches game_outcome predictions to actual results
  - Auto-resolves via resolve_binary()
  - Runs on a schedule (Monday after game day)
  Estimated effort: 3-5 days
  Impact: Enables scoring for the highest-volume prediction category

PRIORITY 2 — EXTRACTION PROMPT TUNING (1 sprint)
  Improve Gemini extraction quality:
  - Add few-shot examples to the prompt (good vs bad extractions)
  - Add explicit "do NOT extract opinions or historical statements" rules
  - Add transcript-specific instructions (handle chunking, speaker diarization)
  - Test with Gemini Pro on a sample to see if quality improves enough to justify cost
  - Build an automated extraction QA pipeline (classify quality, flag low-confidence)
  Estimated effort: 3-5 days
  Impact: Reduces vague/untestable extractions from ~30% to <15%

PRIORITY 3 — DEDUPLICATION ACROSS CHUNKS (0.5 sprint)
  YouTube transcripts get chunked, producing duplicate predictions.
  - Add post-extraction dedup using embedding similarity
  - Or: change chunking strategy to overlap-aware (send adjacent chunks together)
  - Or: use a two-pass approach (extract from full transcript, not chunks)
  Estimated effort: 2-3 days
  Impact: Eliminates ~10-15% duplicate predictions

PRIORITY 4 — STRUCTURED PICK SOURCES (1 sprint)
  Add pick-format sources for easy wins:
  - CBS Sports weekly picks page
  - ESPN consensus picks
  - NFL.com predictions page
  - These publish structured pick tables that are trivially parseable
  Estimated effort: 3-5 days
  Impact: Adds 5-10 pundits with near-100% extraction quality

PRIORITY 5 — BETA LEADERBOARD LAUNCH (1 sprint)
  Build a minimal pundit leaderboard page:
  - Show 10-15 pundits with game_outcome accuracy
  - Display prediction history per pundit
  - Simple "credit score" based on accuracy + volume + timeliness
  - Ship as a page on the existing Next.js dashboard
  Estimated effort: 5 days
  Impact: First user-facing feature. Gets feedback for iteration.

PRIORITY 6 — EXPAND TO 50 PUNDITS (2 sprints)
  With extraction + resolution working:
  - Onboard 40+ more pundits using the YAML config approach
  - Add auto-resolution for draft_pick and trade categories
  - Build monitoring dashboards for extraction quality
  Estimated effort: 2 weeks
  Impact: Reaches critical mass for product viability

BACKLOG (defer until after beta launch):
  - Crowdsourced prediction submission UI
  - Multi-sport expansion (NBA, MLB)
  - Pundit notification system (alert when a prediction is resolved)
  - Historical backfill (scrape past predictions for accuracy history)
  - Monetization integration (premium access to full leaderboard)
""")

print("=" * 70)
print("END OF SPIKE")
print("=" * 70)

### Appendix: Export Raw Data

Save extraction results to CSV for reference and follow-up analysis.

In [ ]:
# Export extraction results to CSV for future reference
output_dir = Path(os.getcwd()) / 'outputs'
output_dir.mkdir(exist_ok=True)

if not extractions_df.empty:
    output_path = output_dir / 'spike_158_extractions.csv'
    extractions_df.to_csv(output_path, index=False)
    print(f"Saved {len(extractions_df)} extractions to {output_path}")
else:
    print("No extractions to export.")

# Summary stats for quick reference
print(f"\n=== Spike #158 Summary ===")
print(f"Date: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
print(f"Pundits tested: {len(TARGET_PUNDITS)}")
print(f"Content pieces fetched: {sum(len(items) for items in ingestion_results.values())}")
print(f"Assertions extracted: {len(extractions_df) if not extractions_df.empty else 0}")
if not extractions_df.empty and 'quality' in extractions_df.columns:
    is_dup_col = 'is_duplicate' in extractions_df.columns
    clean = extractions_df[
        extractions_df['quality'].isin(['clearly_testable', 'conditionally_testable'])
        & ~extractions_df['is_historical']
    ]
    if is_dup_col:
        clean = clean[~clean['is_duplicate']]
    print(f"Clean (usable) assertions: {len(clean)}")
    print(f"Extraction quality score: {len(clean)/len(extractions_df)*100:.0f}%")
print(f"Ingestion errors: {len(ingestion_errors)}")
print(f"Extraction errors: {sum(len(e) for e in extraction_errors.values())}")